In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cust_schema = """
    customer_id STRING,
    email STRING,
    first_name STRING,
    last_name STRING,
    gender STRING,
    street STRING,
    city STRING,
    country_code STRING,
    row_status STRING,
    row_time TIMESTAMP
"""

def process_deletes(microBatchDF, batchId):    
    window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())
    (
        microBatchDF.filter(F.col('row_status') == "delete")
            .withColumn("rank", F.rank().over(window))
            .filter(F.col('rank') == 1)
            .drop("rank")
            .createOrReplaceTempView("deletes")
    )
    
    microBatchDF.sparkSession.sql(
        """
        DELETE FROM dev.silver.customers_orders
        WHERE customer_id in (
            SELECT customer_id FROM deletes
        )    
        """
    )
    sql_query = (
        """
            MERGE INTO dev.silver.delete_requests r
            USING deletes d
            ON r.customer_id = d.customer_id
            WHEN MATCHED THEN 
                UPDATE SET 
                    status = 'deleted'
        """
    )
    microBatchDF.sparkSession.sql(sql_query)

def process_delete_requests():
    deltes_df = (
        spark.readStream
            .table('dev.bookstore_bronze.bookstore_bronze')
            .filter("topic = 'customers'")
            .select(F.from_json(F.col("value").cast('string'), schema=cust_schema).alias("data"))
            .select("data.*", F.col('data.row_time').alias("request_timestamp"))
            .filter(F.col("row_status") == "delete")
            .select(
                "customer_id", 
                "request_timestamp", 
                F.date_add("request_timestamp", 30).alias("deadline"),
                F.lit("requested").alias("status")
            )
            .writeStream
               .foreachBatch(process_deletes)
               .option("checkpointLocation", "dbfs:/Volumes/dev/landing_zone/kafka_source/checkpoints/delete_requestes")
               .trigger(availableNow=True)
               .start()
    )